<a href="https://colab.research.google.com/github/igorfc211-source/PythonAiIMDB/blob/main/ImdbProjetoPython.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub
import kagglehub
import os


from google.colab import userdata
userdata.get('ImdbAPI')

os.environ["GOOGLE_API_KEY"] = userdata.get('ImdbAPI')

# Download latest version
path = kagglehub.dataset_download("danielgrijalvas/movies")


print("Path to dataset files:", path)
import pandas as pd


# Carregar o CSV
df = pd.read_csv(f"{path}/movies.csv")


movie = df["name"].tolist()

Using Colab cache for faster access to the 'movies' dataset.
Path to dataset files: /kaggle/input/movies


In [ ]:
# !pip install googletrans==4.0.0-rc1


In [ ]:
# from googletrans import Translator
# #tradução
# #faz isso logo
# translator = Translator()
# texto = "Hello, how are you?"

# traducao = translator.translate(texto, src="en", dest="pt")

# print("Texto original:", texto)
# print("Idioma detectado:", traducao.src)
# print("Tradução:", traducao.text)


In [ ]:
from datetime import datetime

#selecionando as colunas
df_filmes = df[['year', 'name', 'genre', 'score']]

print(df_filmes.head(10))


#----------------------Chamando API Gemini--------------------------



   year                                            name      genre  score
0  1980                                     The Shining      Drama    8.4
1  1980                                 The Blue Lagoon  Adventure    5.8
2  1980  Star Wars: Episode V - The Empire Strikes Back     Action    8.7
3  1980                                       Airplane!     Comedy    7.7
4  1980                                      Caddyshack     Comedy    7.3
5  1980                                 Friday the 13th     Horror    6.4
6  1980                              The Blues Brothers     Action    7.9
7  1980                                     Raging Bull  Biography    8.2
8  1980                                     Superman II     Action    6.8
9  1980                                 The Long Riders  Biography    7.0


Teste com sklearning

In [ ]:
!pip install fuzzywuzzy


In [ ]:
from datetime import datetime
import pandas as pd
from fuzzywuzzy import process
import time
from google import genai

# Criar cliente do Gemini
client = genai.Client()
chat = client.chats.create(model="gemini-2.5-flash")

# Exemplo: dataframe com filmes
# df = pd.read_csv("movies.csv")  # supondo que você já tenha carregado
df_filmes = df[['year', 'name', 'genre', 'score']]

# >---------- selecionar os filmes ----------<
entrada = []
for i in range(4):
    consulta = input(f"\n \b Digite o nome do filme {i+1}: ")
    resultados = process.extract(consulta, df_filmes["name"].tolist(), limit=5)

    print("\nOpções encontradas:")
    for idx, resultado in enumerate(resultados, start=1):
        print(f"{idx}. {resultado[0]} (similaridade: {resultado[1]})")

    escolha = int(input("Escolha o número do filme correto: "))
    entrada.append(resultados[escolha - 1][0])

print("\nSua lista final de filmes:", entrada)

# --------------- Processamento com fuzzy/gemini ---------
print("\nGerando recomendações, aguarde...")
time.sleep(2)

filmes_escolhidos = []
for filme in entrada:
    match = process.extractOne(filme, df_filmes["name"].tolist())
    if match:
        filmes_escolhidos.append(match[0])

perfil = df_filmes[df_filmes["name"].isin(filmes_escolhidos)]
generos_preferidos = perfil["genre"].value_counts().index.tolist()
score_medio = perfil["score"].mean()
#---------definindo os recomendados e organizando por avalaiação-----------

recomendados = df_filmes[
    (~df_filmes["name"].isin(filmes_escolhidos)) &
    (df_filmes["genre"].isin(generos_preferidos)) &
    (df_filmes["score"] >= score_medio)
].sort_values(by="score", ascending=False)

#---------------printa os recomendados------------------
print("\n--- Recomendações automáticas ---")
print(recomendados[["name", "genre", "score"]].head(10))

# --- recomendação com o Gemini ---
print("\n--- Recomendações do Gemini ---")
prompt = f"""
Sou um sistema de recomendação de filmes.
O usuário escolheu os filmes: {', '.join(filmes_escolhidos)}.

Com base nesses títulos, recomende 10 filmes que não foram escolhidos pelo usuário,
que o usuário provavelmente vai gostar, levando em conta diretor, gênero, o ano, e o contexto do filme, por exemplo, se for filme de herói,
recomende de herois, se for de ficcção cientifica e por aí vai.
e diga apenas o nome e o genero do filme, sem explicações


"""

resposta = chat.send_message(prompt)

print(resposta.text)



  Digite o nome do filme 1: lala

Opções encontradas:
1. The Postman Always Rings Twice (similaridade: 68)
2. Irreconcilable Differences (similaridade: 68)
3. Alamo Bay (similaridade: 68)
4. Fatal Attraction (similaridade: 68)
5. Salaam Bombay! (similaridade: 68)
Escolha o número do filme correto: 2

  Digite o nome do filme 2: lorenzos

Opções encontradas:
1. O (similaridade: 90)
2. Lore (similaridade: 90)
3. Lorenzo's Oil (similaridade: 79)
4. Rent (similaridade: 68)
5. Open (similaridade: 68)
Escolha o número do filme correto: 3

  Digite o nome do filme 3: lala

Opções encontradas:
1. The Postman Always Rings Twice (similaridade: 68)
2. Irreconcilable Differences (similaridade: 68)
3. Alamo Bay (similaridade: 68)
4. Fatal Attraction (similaridade: 68)
5. Salaam Bombay! (similaridade: 68)
Escolha o número do filme correto: 4

  Digite o nome do filme 4: rock

Opções encontradas:
1. Rocky III (similaridade: 90)
2. Rock & Rule (similaridade: 90)
3. Body Rock (similaridade: 90)
4.